In [1]:
import insightface
import numpy as np
import faiss
import cv2
import pickle
import os
import glob
# from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
import threading
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import json
import time


In [2]:
# GLOBAL LOCK (thread safe FAISS)
lock = threading.Lock()

# ---------------------------
# FAST FACE ENGINE
# ---------------------------
class FaceRecognizer:

    def __init__(self):
        # FASTEST MODEL (instead of buffalo_l)
        self.app = insightface.app.FaceAnalysis(
            name="buffalo_l",   
            providers=['CPUExecutionProvider'],
            allowed_modules=['detection', 'recognition','genderage']
        )
        self.app.prepare(
            ctx_id=0,
            det_size=(640,640)  # optimal speed/accuracy balance
        )

    def get_faces_data(self, image_path):
        img = cv2.imread(image_path)
        if img is None:
            return None, None
        faces = self.app.get(img)
        return img, faces

# ---------------------------
# ULTRA FAST FAISS DB
# ---------------------------
class FaceDB:
    
    def __init__(self, dim=512):
        self.dim = dim
        # MUCH faster than IndexFlatIP
        # self.index = faiss.IndexHNSWFlat(dim, 32)
        # self.index.hnsw.efSearch = 64
        # self.index.hnsw.efConstruction = 40
        self.index = faiss.IndexFlatIP(dim)  # Inner Product (Cosine Similarity)
        self.id_map = []

    def add(self, embedding, person_id):
        embedding = np.array([embedding]).astype('float32')
        faiss.normalize_L2(embedding)
        self.index.add(embedding)
        self.id_map.append(person_id)

    def search(self, embedding, threshold=0.3):
        if self.index.ntotal == 0:
            return None
        embedding = np.array([embedding]).astype('float32')
        faiss.normalize_L2(embedding)
        D, I = self.index.search(embedding, k=1)
        similarity = D[0][0]
        if similarity > threshold:
            return self.id_map[I[0][0]]
        return None

    def save(self, folder_path):
        """Saves both the index and the ID map to the specified folder."""
        os.makedirs(folder_path, exist_ok=True)
        
        # 1. Save the FAISS index
        faiss.write_index(self.index, os.path.join(folder_path, "face_index.bin"))
        
        # 2. Save the ID mapping list
        with open(os.path.join(folder_path, "id_map.pkl"), "wb") as f:
            pickle.dump(self.id_map, f)
        # print(f"Database saved to {folder_path}")
        
    def load(self, folder_path):
        """Loads the index and ID map from the specified folder."""
        index_file = os.path.join(folder_path, "face_index.bin")
        map_file = os.path.join(folder_path, "id_map.pkl")
        
        if os.path.exists(index_file) and os.path.exists(map_file):
            # 1. Load the FAISS index
            self.index = faiss.read_index(index_file)
            # 2. Load the ID mapping list
            with open(map_file, "rb") as f:
                self.id_map = pickle.load(f)
            # print(f"Database loaded successfully from {folder_path}")
        else:
            print("Error: Database files not found in the specified path.")


################## PATHS
DB_PATH = r"C:\Users\harshit\Desktop\FaceDetection\DB"
FACES_SAVE_PATH = r"C:\Users\harshit\Desktop\FaceDetection\DetectedFaces"
RAW_PATH = r"C:\Users\harshit\Desktop\FaceDetection\RawImage"

os.makedirs(FACES_SAVE_PATH, exist_ok=True)
os.makedirs(DB_PATH, exist_ok=True)

# ---------------------------
# LOAD ONCE (NOT EVERY IMAGE)
# ---------------------------
face_db = FaceDB()
face_engine = FaceRecognizer()

# ---------------------------
# FAST PROCESS FUNCTION
# ---------------------------
# 1. Load the existing FAISS database
if os.path.exists(os.path.join(DB_PATH, "face_index.bin")):
    face_db.load(DB_PATH)
    
def process_image(image_path):
    img_name = image_path.split("\\")[-1]
    # print("Processing:", img_name)

    img, faces = face_engine.get_faces_data(image_path)
    if img is None or faces is None or len(faces) == 0:
        return False, []
         
    # print(f"{img_name}- number of faces detected: {len(faces)}")
    found_in_this_image = []
    
    # Chec for each face detected inside a Image
    for face in faces:
        if face.det_score < 0.7:
            print(f"Skipping: Could not read face from image at {image_path}")
            continue
        
        # The embedding logic 
        emb = face.embedding
        new_face_detected = False

        # FAISS database operation
        with lock:
            # 1. Search the face in database
            person = face_db.search(emb)
            # 2. Store the embedding in the FAISS index
            if person is None:
                person = f"person_{len(face_db.id_map)}"
                face_db.add(emb, person)
                # 3. Save FAISS changes back to disk
                face_db.save(DB_PATH)
                new_face_detected = True
            else:
                print(f"Recognized: {person}  in Image-{img_name}")
            found_in_this_image.append(person)
            # print(f"{img_name} out")

        # 3. IO Operations (Outside the lock to keep it fast)
        if new_face_detected:
            # Extract gender and age
            gender = "Male" if face.gender == 1 else "Female"
            age = int(face.age)
            
            # 2. Store the physical face image
            bbox = face.bbox.astype(int)
            y1,y2 = max(0,bbox[1]), min(img.shape[0],bbox[3])
            x1,x2 = max(0,bbox[0]), min(img.shape[1],bbox[2])
            face_crop = img[y1:y2,x1:x2]
            
            if face_crop.size > 0:
                cv2.imwrite(os.path.join(FACES_SAVE_PATH,f"{person}.jpg"), face_crop)
                # print(f"New person: {person} - Embedding stored & Face image saved from Image-{img_name}")
            
            # Save metadata JSON
            metadata = {
                "person_id": person,
                "gender": gender,
                "age": age}
            json_path = os.path.join(FACES_SAVE_PATH, f"{person}.json")
            with open(json_path, "w") as f:
                json.dump(metadata, f, indent=4)
            # print(f"New person: {person}, {gender}, Age: {age}")
    return img_name, found_in_this_image

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: C:\Users\harshit/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: C:\Users\harshit/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\harshit/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\harshit/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\harshit/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [3]:
# ---------------------------
# PARALLEL EXECUTION
# ---------------------------
def run_parallel():
    # Use glob to find all files matching common image extensions
    files = glob.glob(os.path.join(RAW_PATH,'*.jpg')) + \
            glob.glob(os.path.join(RAW_PATH,'*.png'))
    cpu_count = os.cpu_count()
    print(cpu_count)
    
    # IMPORTANT: Don't use full CPU (avoid overload)
    workers = max(1, cpu_count // 2) +2
    print("Using workers:", workers)
    with ThreadPoolExecutor(max_workers=workers) as executor:
        result = list(executor.map(process_image, files))
        # print(f"#### {result}")
    return result

if __name__ == "__main__":
    start_time = time.perf_counter()
    results = run_parallel()
    
    json_path = os.path.join(FACES_SAVE_PATH, f"Final_mapping.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=4)
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time:.4f} seconds")


8
Using workers: 6
Recognized: person_0  in Image-IMG_4177.JPG
Recognized: person_3  in Image-IMG_4176.JPG
Recognized: person_0  in Image-IMG_4176.JPG
Recognized: person_0  in Image-CLRV7922.JPG
Recognized: person_5  in Image-GYGM0854.JPG
Recognized: person_4  in Image-GYGM0854.JPG
Recognized: person_0  in Image-GYGM0854.JPG
Recognized: person_6  in Image-GYGM0854.JPG
Recognized: person_0  in Image-IMG_4178.JPG
Recognized: person_3  in Image-IMG_4178.JPG
Recognized: person_3  in Image-IMG_4180 - Copy (2).JPG
Recognized: person_0  in Image-IMG_4180 - Copy (2).JPG
Recognized: person_3  in Image-IMG_4183 - Copy (3).JPG
Recognized: person_0  in Image-IMG_4183 - Copy (3).JPG
Recognized: person_3  in Image-IMG_4185 - Copy - Copy (2).JPG
Recognized: person_0  in Image-IMG_4185 - Copy - Copy (2).JPG
Recognized: person_3  in Image-IMG_4182.JPG
Recognized: person_0  in Image-IMG_4182.JPG
Recognized: person_3  in Image-IMG_4205 - Copy (2).JPG
Recognized: person_0  in Image-IMG_4205 - Copy (2).JPG

In [4]:
# 10 Imgaes: 132 MB
# Small model: Elapsed time: 3.1009 seconds
# Large model: Elapsed time: 6.7717 seconds

# 115 Image: 1.39 GB
# Small model: Elapsed time: 27.0567 seconds
# Large model: Elapsed time: 85.3933 seconds - 70.8855 second

In [5]:
# Check if the directory exists and remove it
cleanup = False
if cleanup==True:
    import shutil
    DB_PATH = r"C:\Users\harshit\Desktop\FaceDetection\DB"
    FACES_SAVE_PATH = r"C:\Users\harshit\Desktop\FaceDetection\DetectedFaces"
    if os.path.exists(DB_PATH):
        shutil.rmtree(DB_PATH) # Removes the directory and all its contents
        shutil.rmtree(FACES_SAVE_PATH) 
        print(f"FAISS database directory '{DB_PATH}' deleted.")
    else:
        print(f"FAISS database directory '{DB_PATH}' not found.")